# 01 - Preprocessing

Runs the three pipeline steps from `src/` and shows each result inline. The logic lives in `src/` (one source of truth), so this notebook and `python run_pipeline.py all` always do exactly the same thing.

**Before running:** put the dataset CSV in `data/raw/` (it must be the only `.csv` there, or set `raw.file` in `config/config.yaml`).

Run the cells top to bottom.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"Project root not found from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import clean, convert, profiling
from src import settings as S

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
print("Project root:", ROOT)

Project root: c:\Universiti Malaya (Extra)\DATATHON\liqi-datathon


## 1. Convert: CSV -> Parquet (lossless)

Reads the CSV in chunks and stores every column as text, so nothing is altered (leading zeros kept, `NA` not treated as missing). Output: `data/interim/transactions_raw.parquet`.

In [2]:
convert.run()

File      : swift_umdac_datathon_2026_fictitious_payment_data.csv (4,952.2 MB)
Delimiter : ','   Encoding: utf-8-sig   Columns: 21
  rows written: 10,000,000
Saved 10,000,000 rows -> data\interim\transactions_raw.parquet
Column name mapping -> reports\profiling\column_mapping.csv


Original header names and the normalised names used from here on:

In [3]:
pd.read_csv(S.COLUMN_MAPPING_CSV, keep_default_na=False)

,position,original_name,normalised_name
0,0,DISCLAIMER,disclaimer
1,1,EPOCH_SECONDS,epoch_seconds
2,2,CURRENCY_CODE,currency_code
3,3,USD_AMOUNT,usd_amount
4,4,ORDERING_BIC8,ordering_bic8
5,5,BENEFICIARY_BIC8,beneficiary_bic8
6,6,BIC8_PATH,bic8_path
7,7,TOTAL_DURATION,total_duration
8,8,CONTINUITY_QFLAG,continuity_qflag
9,9,CURRENCY_CONVERSION_COUNT,currency_conversion_count


## 2. Profile every column

Scans the whole file. Check here: which columns are empty, which look numeric, which have leading zeros (identifiers), and the most frequent values.

In [4]:
profiling.run()

  rows profiled: 10,000,000 / 10,000,000
Rows: 10,000,000   Columns: 21   Exact duplicate rows: 1
                   column  empty_pct distinct  min_len  max_len  numeric  leading_zero
               disclaimer   100.0000        1      506      506        0             0
            epoch_seconds     0.0000   >50000       10       10 10000000             0
            currency_code     0.0000      155        3        3        0             0
               usd_amount     0.0000   >50000        3       14 10000000             0
            ordering_bic8     0.0000     8392        8        8        0       9832750
         beneficiary_bic8     0.0000     8266        8        8        0       9803075
                bic8_path     0.0000   >50000       17      205        0       9832721
           total_duration     0.0000    10151        1        5 10000000             0
         continuity_qflag     0.0000        2        1        1        0             0
currency_conversion_count     0.

In [5]:
profile_summary = pd.read_csv(S.PROFILE_CSV, keep_default_na=False)
profile_summary

,column,rows,empty,empty_pct,non_empty,whitespace_only,padded_with_whitespace,distinct,min_len,max_len,numeric,integer,leading_zero,numeric_min,numeric_max,top_values
0,disclaimer,10000000,9999999,100.0000,1,0,0,1,506,506,0,0,0,,,"""WARNING: This dataset contains purely fictitious data that has been generat..."
1,epoch_seconds,10000000,0,0.0000,10000000,0,0,>50000,10,10,10000000,10000000,0,1682853746.0,1690797361.0,
2,currency_code,10000000,0,0.0000,10000000,0,0,155,3,3,0,0,0,,,'EUR': 3198639 | 'USD': 1702991 | 'GBP': 873781 | 'CHF': 803898 | 'SGD': 367648
3,usd_amount,10000000,0,0.0000,10000000,0,0,>50000,3,14,10000000,0,0,0.01,15664132213.09,
4,ordering_bic8,10000000,0,0.0000,10000000,0,0,8392,8,8,0,0,9832750,,,'0004IMD0': 154396 | '0309CHZ0': 130375 | '0026FIH0': 96204 | '0085BEB0': 89...
5,beneficiary_bic8,10000000,0,0.0000,10000000,0,0,8266,8,8,0,0,9803075,,,'0309CHZ0': 169380 | '0004IMD0': 166133 | '0026FIH0': 124677 | '0085BEB0': 1...
6,bic8_path,10000000,0,0.0000,10000000,0,0,>50000,17,205,0,0,9832721,,,
7,total_duration,10000000,0,0.0000,10000000,0,0,10151,1,5,10000000,10000000,0,2.0,21649.0,'25': 146285 | '23': 146166 | '24': 145990 | '26': 145354 | '21': 144065
8,continuity_qflag,10000000,0,0.0000,10000000,0,0,2,1,1,0,0,0,,,'Y': 9232796 | 'N': 767204
9,currency_conversion_count,10000000,0,0.0000,10000000,0,0,18,1,2,10000000,10000000,0,0.0,17.0,'1': 5895841 | '2': 2017992 | '0': 1086531 | '3': 666605 | '4': 222062


## 3. Clean: assign types and cast

Types are inferred from the full-file profile. The step refuses to save if the row count changes or a cast creates a missing value. Output: `data/processed/transactions_clean.parquet`.

In [6]:
clean.run()

  rows cleaned: 10,000,000 / 10,000,000
                   column       type  missing  missing_pct                                              reason
            epoch_seconds datetime_s        0       0.0000 all values fall in the Unix-seconds range 2000-2100
            currency_code     string        0       0.0000                              free text / identifier
               usd_amount      float        0       0.0000                              all values are numeric
            ordering_bic8     string        0       0.0000                              free text / identifier
         beneficiary_bic8     string        0       0.0000                              free text / identifier
                bic8_path     string        0       0.0000                              free text / identifier
           total_duration        int        0       0.0000                             all values are integers
         continuity_qflag       bool        0       0.0000              

The type chosen for each column and why. **Review this table.** If a type is wrong, or a column should be dropped or renamed, edit `clean.force_types`, `clean.drop_columns` or `clean.rename` in `config/config.yaml`, then re-run the `clean.run()` cell and this one (no kernel restart needed).

In [7]:
pd.read_csv(S.TYPE_PLAN_CSV, keep_default_na=False)

,column,output_name,type,reason
0,disclaimer,,DROPPED,clean.drop_columns
1,epoch_seconds,epoch_seconds,datetime_s,all values fall in the Unix-seconds range 2000-2100
2,currency_code,currency_code,string,free text / identifier
3,usd_amount,usd_amount,float,all values are numeric
4,ordering_bic8,ordering_bic8,string,free text / identifier
5,beneficiary_bic8,beneficiary_bic8,string,free text / identifier
6,bic8_path,bic8_path,string,free text / identifier
7,total_duration,total_duration,int,all values are integers
8,continuity_qflag,continuity_qflag,bool,only Y/N values
9,currency_conversion_count,currency_conversion_count,int,all values are integers


## 4. Load and check the clean data

This loads the whole typed file into memory. If it does not fit, pass `columns=[...]` to `pd.read_parquet` to load only what you need.

In [8]:
df = pd.read_parquet(S.CLEAN_PARQUET)
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB in memory")
df.head(10)

10,000,000 rows x 21 columns, 5,601.8 MB in memory


,epoch_seconds,currency_code,usd_amount,ordering_bic8,beneficiary_bic8,bic8_path,total_duration,continuity_qflag,currency_conversion_count,participant_count,transaction_type_code,ordering_account,beneficiary_account,ordering_details,ordering_tokens,beneficiary_details,beneficiary_tokens,ordering_type,beneficiary_type,anomaly_type,is_anomaly
0,2023-05-29 12:29:04+00:00,USD,500.00,0621US30,0219US30,0621US30;0219US30,4,True,0,2,Direct,_FAKE_00.-0/0092Y9262,_FAKE_0009559K28N,/00.-0/0092Y9262\nSCHULTZ PLC 38626 CHRISTENSEN GARDE\nN 30410 US\n8530276,/<chr:99.-9/9999A9999>\n<line_break:35><business:7> <company_keyword:3> <oth...,/0009559K28N\nS MRDHAA 66 SIRAJROAD 2711 GAAIBAAN\nDHAA BD,/<chr:9999999A99A>\n<line_break:35><chr:A> <last_name:6> <chr:99> <street:9>...,B,C,NaN,0
1,2023-06-13 12:25:45+00:00,TRY,1390.34,0024TRI0,0380CNB0,0024TRI0;0380CNB0,396,True,1,2,Direct,_FAKE_TR53002400000000890222,_FAKE_0000000008A79136,/TR53002400000000890222\nCALIK 1162 SERAFEDDIN VILLE 63439 T\nR,/<iban:22>\n<line_break:35><business:5> <chr:9999> <street:16> <post_code:TR...,//0000000008A79136\nGUI LAN XIE DZUO TAO LU CN,/<separator:/><chr:9999999999A99999>\n<first_name:8> <other_words:13> <chr:A...,B,C,NaN,0
2,2023-07-04 17:38:32+00:00,GBP,3972.37,0007JEH0,0295GB20,0007JEH0;0295GB20,3,True,0,2,Direct,_FAKE_00 000 000B0623 133W,_FAKE_GB30029500000000623158,"/00 000 000B0623 133W\n1/GORDON, SMITH AND MARTIN\n2/BROWN PORT 536\n3/JE/AL...",/<chr:99 999 999A9999 999A>\n<other_words:26>\n2/<street:10> <chr:999>\n3/<c...,/GB30029500000000623158\nW GUI XIANG YZUO XIANG GANG JIE\nHK,/<iban:22>\n<chr:A> <last_name:10> <chr:AAAA > <street:15> <country_code:HK>,B,C,NaN,0
3,2023-05-19 03:06:09+00:00,SGD,1920.69,0203SGS0,0037DEF0,0203SGS0;0405US30;0037DEF0,4341,True,2,3,Indirect,_FAKE_00000 F00/08/57/6/40,_FAKE_DE49003700000000292310,/00000 F00/08/57/6/40\nPERUM WASKITA JL. DIPENOGORO 2 2259\n7 SG,/<chr:99999 A99/99/99/9/99>\n<line_break:35><business:5> <company_keyword:7>...,/DE49003700000000292310\nM ALBA 80036 CIRCUNVALACION ANDORRA\nSAN TIMOTEO DE...,/<iban:22>\n<other_words:6> <last_name:4> <chr:99999> <street:22> <city:MX:2...,B,C,NaN,0
4,2023-06-05 21:32:56+00:00,EUR,22130.99,0017DEF0,0087ZAJ0,0017DEF0;0087ZAJ0,20,True,1,2,Direct,_FAKE_DE13001700000000291498,_FAKE_ 0000000.00099.3989,/DE13001700000000291498\nKREUSEL 1-9 ANA-BIRNBAUM-RING FINST\nERWALDE DE,/<iban:22>\n<line_break:35><business:7> <chr:9-9> <street:17> <city:DE:12> <...,/ 0000000.00099.3989\nMHWRSZN YRN KHWDRW SHHRSTN `LY,/<chr: 9999999.99999.9999>\n<business:18> <city:AF:11>,B,B,NaN,0
5,2023-06-12 19:21:22+00:00,EUR,349.95,0001MQM0,0009TNT0,0001MQM0;0009TNT0,1588,True,1,2,Direct,_FAKE_00E69 0N 873,_FAKE_TN11000900000000886688,/00E69 0N 873\nLE,/<chr:99A99 9A 999>\n<business:2>,//TN11000900000000886688\nBIN BAO ZHONG QING JIE VZUO 971\n012 MA AN SHA...,/<separator:/><iban:22>\n<line_break:35><first_name:4> <last_name:4> <street...,B,C,NaN,0
6,2023-06-13 15:03:59+00:00,EUR,1447.90,0412FRP0,0031TWT0,0412FRP0;0023FIH0;0035LUL0;0071KRS0;0005BHB0;0031TWT0,1900,True,3,6,Covered,_FAKE_FR45041200000000349730,_FAKE_89V7929,/FR45041200000000349730\nMAILLET BOULEVARD CHARRIER 38793 LE\nVEQUE FR,/<iban:22>\n<line_break:35><business:7> <street:18> <post_code:FR:5> <city:F...,/89V7929\nANDRE SA PAUL\n9342,/<chr:99A9999>\n<business:5> <company_keyword:2> <city:CM:4>\n<numerical:4>,B,B,NaN,0
7,2023-06-20 08:07:27+00:00,EUR,12087.80,0016AT20,0047CHZ0,0016AT20;0047CHZ0,24,True,1,2,Direct,_FAKE_AT60001600000000018393,_FAKE_CH74004700000000092030,/AT60001600000000018393\nPOINTNER & SCHANDL KG 4/2 WILHELMPL\nATZ AT,/<iban:22>\n<line_break:35><business:21> <chr:9/9> <street:12> <country_code...,/CH74004700000000092030\n1/ROSTISLAV BABKO\n2/8 NADIYI PUCHKOVS'KOYI 01793\n...,/<iban:22>\n<other_words:17>\n2/<chr:9> <street:20> <post_code:UA:5>\n3/<cou...,B,C,NaN,0
8,2023-07-25 04:31:36+00:00,EUR,2345.31,0001RUM0,0127IT20,0001RUM0;0143SGS0;0127IT20,23,True,2,3,Covered,_FAKE_0-0E0-000J840805,_FAKE_IT710127000000008

In [9]:
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(4),
    "distinct": df.nunique(),
})

,dtype,missing,missing_pct,distinct
epoch_seconds,"datetime64[us, UTC]",0,0.0000,5085929
currency_code,str,0,0.0000,155
usd_amount,float64,0,0.0000,3464639
ordering_bic8,str,0,0.0000,8392
beneficiary_bic8,str,0,0.0000,8266
bic8_path,str,0,0.0000,4403689
total_duration,int64,0,0.0000,10151
continuity_qflag,bool,0,0.0000,2
currency_conversion_count,int64,0,0.0000,18
participant_count,int64,0,0.0000,19


In [10]:
df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
usd_amount,10000000.0,97204.102038,9.909715e+06,0.01,634.0875,3661.895,19959.855,1.566413e+10
total_duration,10000000.0,771.510208,1.228202e+03,2.00,26.0000,56.000,1105.000,2.164900e+04
currency_conversion_count,10000000.0,1.343024,9.446476e-01,0.00,1.0000,1.000,2.000,1.700000e+01
participant_count,10000000.0,2.581571,9.592645e-01,2.00,2.0000,2.000,3.000,2.200000e+01
is_anomaly,10000000.0,0.000701,2.645958e-02,0.00,0.0000,0.000,0.000,1.000000e+00


Label balance (only if the label column from `config.yaml` exists):

In [11]:
label = S.load_config()["clean"]["label_column"]
if "is_anomaly" in df.columns:
    print(df["is_anomaly"].value_counts().rename({0: "normal", 1: "anomaly"}).to_string())
    print(f"\nAnomaly rate: {df['is_anomaly'].mean() * 100:.4f}%\n")
    display(df[label].value_counts(dropna=False).to_frame("rows"))
else:
    print("is_anomaly was not created - check clean.label_column in config/config.yaml")

is_anomaly
normal     9992994
anomaly       7006

Anomaly rate: 0.0701%



,rows
anomaly_type,
NaN,9992994
ROMANCE_SCAM,2476
FAT_FINGER_MISTAKE,1940
CEO_SCAM,945
DUPLICATE_PAYMENT,679
ACCOUNT_REPLACEMENT_SCAM,649
CUSTOMER_RECALL,317
